# Thesis — Improved 7-Model Singular Comparison

All 7 dual-branch variants trained through one upgraded shared harness
(AdamW + warmup->cosine + label smoothing + SWA + best-by-val-F1 + AMP), with a single
shared Optuna search on the Full Model whose hyperparameters are locked and reused by all 7
(fair architecture comparison). Image-only (clinical + dermoscopic), custom 70/15/15 split.
Toggle DATASET_DIR for Derm7pt or Milk10k; artifacts suffixed `_improved_{dataset}`.

## 1 — Setup

In [ ]:
import os, copy, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torchvision.models as models
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, f1_score, accuracy_score, balanced_accuracy_score,
    cohen_kappa_score, matthews_corrcoef, log_loss
)

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 100

# Reproducibility
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('mps' if torch.backends.mps.is_available() else
                      'cuda' if torch.cuda.is_available() else 'cpu')
if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True
print(f'Device: {device}')

import json, math
from torch.optim.swa_utils import AveragedModel, SWALR
import optuna
from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner
optuna.logging.set_verbosity(optuna.logging.WARNING)
print('optuna', optuna.__version__)

### Dataset selector

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║                          DATASET SELECTOR                                ║
# ║   Change this single line to switch between Derm7pt and MILK10k.         ║
# ╚══════════════════════════════════════════════════════════════════════════╝

DATASET_DIR = Path('dataset') / 'Derm7pt'    # or Path('dataset') / 'Milk10k'

DATASET_NAME = DATASET_DIR.name.lower()
assert DATASET_NAME in ('derm7pt', 'milk10k'), \
    f'Unrecognized DATASET_DIR: {DATASET_DIR}. Expected "Derm7pt" or "milk10k".'
print(f'>>> DATASET_DIR  = {DATASET_DIR}')
print(f'>>> DATASET_NAME = {DATASET_NAME}')


In [ ]:
# ── Dataset-specific dataframe assembly ──────────────────────────────────────
if DATASET_NAME == 'derm7pt':
    derm_raw = pd.read_csv(DATASET_DIR / 'meta' / 'meta.csv')
    derm_raw['clinic_path'] = derm_raw['clinic'].apply(lambda x: str(DATASET_DIR / 'images' / x))
    derm_raw['derm_path']   = derm_raw['derm'].apply(lambda x: str(DATASET_DIR / 'images' / x))
    derm_raw['diagnosis']   = derm_raw['diagnosis'].str.strip().str.lower()
    diagnosis_groups = {
        'melanoma': 'MEL', 'melanoma (less than 0.76 mm)': 'MEL', 'melanoma (in situ)': 'MEL',
        'melanoma (0.76 to 1.5 mm)': 'MEL', 'melanoma (more than 1.5 mm)': 'MEL',
        'melanoma metastasis': 'MEL',
        'clark nevus': 'NV', 'reed or spitz nevus': 'NV', 'dermal nevus': 'NV',
        'blue nevus': 'NV', 'congenital nevus': 'NV', 'combined nevus': 'NV',
        'recurrent nevus': 'NV',
        'basal cell carcinoma': 'BCC',
        'seborrheic keratosis': 'SK',
        'lentigo': 'MISC', 'dermatofibroma': 'MISC', 'vascular lesion': 'MISC',
        'melanosis': 'MISC', 'miscellaneous': 'MISC',
    }
    derm_raw['diagnosis_group'] = derm_raw['diagnosis'].map(diagnosis_groups)
    dataset_df = derm_raw[['clinic_path', 'derm_path', 'diagnosis_group']].copy()
else:  # milk10k
    milk_meta = pd.read_csv(DATASET_DIR / 'MILK10k_Training_Metadata.csv')
    milk_gt   = pd.read_csv(DATASET_DIR / 'MILK10k_Training_GroundTruth.csv')
    milk_class_map = {'MEL': 'MEL', 'NV': 'NV', 'BCC': 'BCC', 'BKL': 'SK',
                      'DF': 'MISC', 'VASC': 'MISC'}
    drop_classes = {'AKIEC', 'SCCKA', 'INF', 'BEN_OTH', 'MAL_OTH'}
    gt_cols = [c for c in milk_gt.columns if c != 'lesion_id']
    milk_gt['raw_class'] = milk_gt[gt_cols].idxmax(axis=1)
    milk_gt = milk_gt[~milk_gt['raw_class'].isin(drop_classes)].copy()
    milk_gt['diagnosis_group'] = milk_gt['raw_class'].map(milk_class_map)
    clinic_meta = (milk_meta[milk_meta['image_type'] == 'clinical: close-up']
                   [['lesion_id', 'isic_id']].rename(columns={'isic_id': 'clinic_isic'}))
    derm_meta   = (milk_meta[milk_meta['image_type'] == 'dermoscopic']
                   [['lesion_id', 'isic_id']].rename(columns={'isic_id': 'derm_isic'}))
    paths = clinic_meta.merge(derm_meta, on='lesion_id')
    milk_raw = milk_gt[['lesion_id', 'diagnosis_group']].merge(paths, on='lesion_id')
    milk_raw['clinic_path'] = milk_raw.apply(
        lambda r: str(DATASET_DIR / 'MILK10k_Training_Input' / r['lesion_id'] / f"{r['clinic_isic']}.jpg"), axis=1)
    milk_raw['derm_path'] = milk_raw.apply(
        lambda r: str(DATASET_DIR / 'MILK10k_Training_Input' / r['lesion_id'] / f"{r['derm_isic']}.jpg"), axis=1)
    dataset_df = milk_raw[['clinic_path', 'derm_path', 'diagnosis_group']].copy()

# 5-class label space (fixed across both datasets so checkpoints remain comparable)
class_names = ['BCC', 'MEL', 'MISC', 'NV', 'SK']
label_map   = {name: i for i, name in enumerate(class_names)}
dataset_df['label'] = dataset_df['diagnosis_group'].map(label_map)
dataset_df = dataset_df.dropna(subset=['label']).reset_index(drop=True)
print(f'{DATASET_NAME}: {len(dataset_df)} samples | {len(class_names)} classes ({class_names})')


class SkinLesionDualDataset(Dataset):
    """Returns (clinic_img, derm_img, label) for each sample."""
    def __init__(self, dataframe, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        clinic_img = Image.open(row['clinic_path']).convert('RGB')
        derm_img   = Image.open(row['derm_path']).convert('RGB')
        if self.transform:
            clinic_img = self.transform(clinic_img)
            derm_img   = self.transform(derm_img)
        return clinic_img, derm_img, torch.tensor(row['label'], dtype=torch.long)


# ── Transforms ───────────────────────────────────────────────────────────────
train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
val_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# ── Stratified 70/15/15 split + class weights ────────────────────────────────
def build_dataset(df_in, name):
    labels  = df_in['label'].values.astype(int)
    indices = np.arange(len(df_in))
    tr, tmp = train_test_split(indices, test_size=0.30, stratify=labels, random_state=SEED)
    va, te  = train_test_split(tmp,     test_size=0.50, stratify=labels[tmp], random_state=SEED)
    train_labels = labels[tr]
    class_counts = np.bincount(train_labels, minlength=len(class_names))
    safe_counts  = np.where(class_counts == 0, 1, class_counts)
    sample_w     = (1.0 / safe_counts)[train_labels]
    loss_w       = torch.tensor(
        [len(train_labels) / (len(class_names) * c) if c > 0 else 0.0 for c in class_counts],
        dtype=torch.float32).to(device)
    return {
        'name': name, 'df': df_in,
        'train_idx': tr, 'val_idx': va, 'test_idx': te,
        'sample_weights': sample_w,
        'loss_weights':   loss_w,
        'class_counts':   class_counts,
    }

ACTIVE_DS = build_dataset(dataset_df, DATASET_NAME)
print(f'{ACTIVE_DS["name"]:>8s}: train={len(ACTIVE_DS["train_idx"])} '
      f'val={len(ACTIVE_DS["val_idx"])} test={len(ACTIVE_DS["test_idx"])} '
      f'| class counts (train) = {ACTIVE_DS["class_counts"].tolist()}')

# ── DataLoaders ──────────────────────────────────────────────────────────────
_NUM_WORKERS = 0 if device.type in ('mps', 'cpu') else min(8, os.cpu_count() or 4)  # Windows/CUDA gets workers
_PIN_MEMORY  = device.type == 'cuda'

def make_loaders(batch_size, ds):
    """Return (train_loader, val_loader, test_loader, test_ds) for a dataset dict."""
    train_ds = SkinLesionDualDataset(ds['df'].iloc[ds['train_idx']], train_transform)
    val_ds   = SkinLesionDualDataset(ds['df'].iloc[ds['val_idx']],   val_transform)
    test_ds  = SkinLesionDualDataset(ds['df'].iloc[ds['test_idx']],  val_transform)
    samp = WeightedRandomSampler(ds['sample_weights'], len(ds['sample_weights']), replacement=True)
    kw = dict(num_workers=_NUM_WORKERS, pin_memory=_PIN_MEMORY,
              persistent_workers=(_NUM_WORKERS > 0))
    train_loader = DataLoader(train_ds, batch_size=batch_size, sampler=samp, drop_last=True, **kw)
    val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False, **kw)
    test_loader  = DataLoader(test_ds,  batch_size=batch_size, shuffle=False, **kw)
    return train_loader, val_loader, test_loader, test_ds


## 2 — Backbone, constants, model classes

In [ ]:
class ResNet50Backbone(nn.Module):
    """ResNet50 feature extractor — returns spatial feature maps [B, 2048, 7, 7]."""
    def __init__(self, pretrained=True):
        super().__init__()
        base = models.resnet50(weights=models.ResNet50_Weights.DEFAULT if pretrained else None)
        self.features = nn.Sequential(
            base.conv1, base.bn1, base.relu, base.maxpool,
            base.layer1, base.layer2, base.layer3, base.layer4)

    def forward(self, x):
        return self.features(x)


# ── Fixed hyperparameters ────────────────────────────────────────────────────
LR             = 0.000750
DROPOUT        = 0.32
WEIGHT_DECAY   = 0.000957
BATCH_SIZE     = 64
FINAL_EPOCHS   = 60
PATIENCE       = 20
UNFREEZE_EPOCH = 5
GRAD_CLIP      = 1.0

# ── Fixed structural constants ──
LR             = 0.000750
DROPOUT        = 0.32
WEIGHT_DECAY   = 0.000957
BATCH_SIZE     = 64
FINAL_EPOCHS   = 60
PATIENCE       = 20
UNFREEZE_EPOCH = 5
GRAD_CLIP      = 1.0
WARMUP_EPOCHS  = 3
SWA_START_FRAC = 0.75
USE_AMP        = (device.type == 'cuda')   # mixed precision, CUDA only (no-op on MPS/CPU)
NC = len(class_names)

# ── SMOKE: fast end-to-end validation (subset + few epochs). Set False for the real run. ──
SMOKE = False
if SMOKE:
    FINAL_EPOCHS = 3
    PATIENCE = 9
    for _k in ('train_idx', 'val_idx', 'test_idx'):
        ACTIVE_DS[_k] = ACTIVE_DS[_k][:128]
    _tl = ACTIVE_DS['df']['label'].values[ACTIVE_DS['train_idx']].astype(int)
    _cc = np.bincount(_tl, minlength=len(class_names)); _safe = np.where(_cc == 0, 1, _cc)
    ACTIVE_DS['sample_weights'] = (1.0 / _safe)[_tl]
    ACTIVE_DS['loss_weights'] = torch.tensor(
        [len(_tl) / (len(class_names) * c) if c > 0 else 0.0 for c in _cc],
        dtype=torch.float32).to(device)
    ACTIVE_DS['class_counts'] = _cc
    print('SMOKE on: subset train/val/test to <=128')

### Model classes (verbatim from the baseline notebook)

In [ ]:
class SingleBranchRGBClassifier(nn.Module):
    """Single-branch baseline: RGB (clinical) image only.
    Keeps the (clinic, derm) forward signature for compatibility with the shared
    train/validate helpers; `derm_img` is ignored."""
    def __init__(self, num_classes=5, dropout=0.32, pretrained=True):
        super().__init__()
        self.resnet_clinic = ResNet50Backbone(pretrained=pretrained)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(
            nn.Linear(2048, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(512, num_classes),
        )

    def _freeze_backbones(self):
        for p in self.resnet_clinic.parameters():
            p.requires_grad = False
        for p in self.resnet_clinic.features[7].parameters():
            p.requires_grad = True

    def unfreeze_resnets(self):
        for p in self.resnet_clinic.features[6].parameters():
            p.requires_grad = True

    def forward(self, clinic_img, derm_img=None):
        x = self.pool(self.resnet_clinic(clinic_img)).flatten(1)
        return self.classifier(x), None


# Parameter count

In [ ]:
class DualBranchBaseline(nn.Module):
    """Concat fusion baseline: dual ResNet50 → channel-stack → 1x1 conv → FC."""
    def __init__(self, num_classes=5, dropout=0.32, pretrained=True):
        super().__init__()
        self.resnet_clinic = ResNet50Backbone(pretrained=pretrained)
        self.resnet_derm   = ResNet50Backbone(pretrained=pretrained)
        self.fusion = nn.Sequential(
            nn.Conv2d(4096, 1024, kernel_size=1, bias=False),
            nn.BatchNorm2d(1024),
            nn.ReLU(inplace=True),
        )
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(
            nn.Linear(1024, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(512, num_classes),
        )

    def _freeze_backbones(self):
        for m in [self.resnet_clinic, self.resnet_derm]:
            for p in m.parameters():
                p.requires_grad = False
            for p in m.features[7].parameters():
                p.requires_grad = True

    def unfreeze_resnets(self):
        for m in [self.resnet_clinic, self.resnet_derm]:
            for p in m.features[6].parameters():
                p.requires_grad = True

    def forward(self, clinic_img, derm_img):
        feat_c = self.resnet_clinic(clinic_img)
        feat_d = self.resnet_derm(derm_img)
        x = self.fusion(torch.cat([feat_c, feat_d], dim=1))
        x = self.pool(x).flatten(1)
        return self.classifier(x), None

In [ ]:
class SEBlock(nn.Module):
    """Squeeze-and-Excitation: global pool → 2-layer MLP → sigmoid → channel-wise scale."""
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(channels, channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels, bias=False),
        )
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        b, c, _, _ = x.shape
        w = self.sigmoid(self.fc(self.avg_pool(x).view(b, c))).view(b, c, 1, 1)
        return x * w


class SEBottleneck(nn.Module):
    """Wraps a torchvision ResNet bottleneck with an SE block after the last bn."""
    def __init__(self, bottleneck, reduction=16):
        super().__init__()
        self.block = bottleneck
        self.se    = SEBlock(bottleneck.conv3.out_channels, reduction)

    def forward(self, x):
        identity = x
        out = self.block.conv1(x); out = self.block.bn1(out); out = self.block.relu(out)
        out = self.block.conv2(out); out = self.block.bn2(out); out = self.block.relu(out)
        out = self.block.conv3(out); out = self.block.bn3(out)
        out = self.se(out)
        if self.block.downsample is not None:
            identity = self.block.downsample(x)
        out = out + identity
        return self.block.relu(out)


class SEResNet50Backbone(nn.Module):
    """ResNet50 with SE blocks injected into every bottleneck. Output [B, 2048, 7, 7]."""
    def __init__(self, pretrained=True, reduction=16):
        super().__init__()
        base = models.resnet50(weights=models.ResNet50_Weights.DEFAULT if pretrained else None)
        for layer in [base.layer1, base.layer2, base.layer3, base.layer4]:
            for i in range(len(layer)):
                layer[i] = SEBottleneck(layer[i], reduction=reduction)
        self.features = nn.Sequential(
            base.conv1, base.bn1, base.relu, base.maxpool,
            base.layer1, base.layer2, base.layer3, base.layer4)

    def forward(self, x):
        return self.features(x)


class DualBranchSEResNet(nn.Module):
    """SE-ResNet50 × 2 → concat → 1x1 conv → FC. Same head as DualBranchBaseline."""
    def __init__(self, num_classes=5, dropout=0.32, pretrained=True):
        super().__init__()
        self.resnet_clinic = SEResNet50Backbone(pretrained=pretrained)
        self.resnet_derm   = SEResNet50Backbone(pretrained=pretrained)
        self.fusion = nn.Sequential(
            nn.Conv2d(4096, 1024, kernel_size=1, bias=False),
            nn.BatchNorm2d(1024),
            nn.ReLU(inplace=True),
        )
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(
            nn.Linear(1024, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(512, num_classes),
        )

    def _freeze_backbones(self):
        for m in [self.resnet_clinic, self.resnet_derm]:
            for p in m.parameters():
                p.requires_grad = False
            for p in m.features[7].parameters():
                p.requires_grad = True

    def unfreeze_resnets(self):
        for m in [self.resnet_clinic, self.resnet_derm]:
            for p in m.features[6].parameters():
                p.requires_grad = True

    def forward(self, clinic_img, derm_img):
        feat_c = self.resnet_clinic(clinic_img)
        feat_d = self.resnet_derm(derm_img)
        x = self.fusion(torch.cat([feat_c, feat_d], dim=1))
        x = self.pool(x).flatten(1)
        return self.classifier(x), None

In [ ]:
class DualBranchElementwiseFusion(nn.Module):
    """Two-branch fusion by an element-wise operator.
    `fusion='add'` -> feat_c + feat_d ; `fusion='mul'` -> feat_c * feat_d.
    A BatchNorm2d after the fusion keeps the activation scale stable
    (the product of two post-ReLU feature maps can otherwise blow up)."""
    def __init__(self, num_classes=5, dropout=0.32, fusion='add', pretrained=True):
        super().__init__()
        assert fusion in ('add', 'mul')
        self.fusion = fusion
        self.resnet_clinic = ResNet50Backbone(pretrained=pretrained)
        self.resnet_derm   = ResNet50Backbone(pretrained=pretrained)
        self.post_fuse = nn.Sequential(
            nn.BatchNorm2d(2048),
            nn.ReLU(inplace=True),
        )
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(
            nn.Linear(2048, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(512, num_classes),
        )

    def _freeze_backbones(self):
        for m in [self.resnet_clinic, self.resnet_derm]:
            for p in m.parameters():
                p.requires_grad = False
            for p in m.features[7].parameters():
                p.requires_grad = True

    def unfreeze_resnets(self):
        for m in [self.resnet_clinic, self.resnet_derm]:
            for p in m.features[6].parameters():
                p.requires_grad = True

    def forward(self, clinic_img, derm_img):
        feat_c = self.resnet_clinic(clinic_img)
        feat_d = self.resnet_derm(derm_img)
        fused = feat_c + feat_d if self.fusion == 'add' else feat_c * feat_d
        x = self.pool(self.post_fuse(fused)).flatten(1)
        return self.classifier(x), None


# Parameter count

In [ ]:
class CrossAttentionFusion(nn.Module):
    """One direction of cross-attention. Q from `clinic_feat`, K/V from `derm_feat`."""
    def __init__(self, in_dim=2048, embed_dim=256, num_heads=4, dropout=0.32):
        super().__init__()
        self.q_proj = nn.Linear(in_dim, embed_dim)
        self.k_proj = nn.Linear(in_dim, embed_dim)
        self.v_proj = nn.Linear(in_dim, embed_dim)
        self.attn   = nn.MultiheadAttention(embed_dim, num_heads, dropout=dropout,
                                            batch_first=True)
        self.norm1  = nn.LayerNorm(embed_dim)
        self.ffn    = nn.Sequential(nn.Linear(embed_dim, embed_dim * 2), nn.GELU(),
                                    nn.Dropout(dropout), nn.Linear(embed_dim * 2, embed_dim))
        self.norm2  = nn.LayerNorm(embed_dim)
        self.pool   = nn.AdaptiveAvgPool1d(1)

    def forward(self, clinic_feat, derm_feat):
        c = clinic_feat.flatten(2).transpose(1, 2)
        d = derm_feat.flatten(2).transpose(1, 2)
        Q = self.q_proj(c); K = self.k_proj(d); V = self.v_proj(d)
        attn_out, attn_weights = self.attn(Q, K, V, need_weights=True,
                                           average_attn_weights=True)
        x = self.norm1(Q + attn_out)
        x = self.norm2(x + self.ffn(x))
        fused = self.pool(x.transpose(1, 2)).squeeze(-1)
        return fused, attn_weights


class BidirectionalCrossAttnFusion(nn.Module):
    """Two cross-attention blocks (c→d and d→c), concatenated and projected."""
    def __init__(self, in_dim=2048, embed_dim=256, num_heads=4, dropout=0.32):
        super().__init__()
        self.c2d  = CrossAttentionFusion(in_dim, embed_dim, num_heads, dropout)
        self.d2c  = CrossAttentionFusion(in_dim, embed_dim, num_heads, dropout)
        self.proj = nn.Linear(embed_dim * 2, embed_dim)

    def forward(self, clinic_feat, derm_feat):
        fused_c2d, attn_c2d = self.c2d(clinic_feat, derm_feat)
        fused_d2c, attn_d2c = self.d2c(derm_feat, clinic_feat)
        fused = self.proj(torch.cat([fused_c2d, fused_d2c], dim=1))
        return fused, (attn_c2d, attn_d2c)


class DualBranchBiCrossAttn(nn.Module):
    """Dual ResNet50 → bidirectional cross-attention → classifier."""
    def __init__(self, num_classes=5, embed_dim=256, num_heads=4, dropout=0.32,
                 pretrained=True):
        super().__init__()
        self.resnet_clinic = ResNet50Backbone(pretrained=pretrained)
        self.resnet_derm   = ResNet50Backbone(pretrained=pretrained)
        self.fusion        = BidirectionalCrossAttnFusion(2048, embed_dim, num_heads, dropout)
        self.classifier = nn.Sequential(
            nn.Linear(embed_dim, embed_dim // 2),
            nn.BatchNorm1d(embed_dim // 2),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(embed_dim // 2, num_classes),
        )

    def _freeze_backbones(self):
        for m in [self.resnet_clinic, self.resnet_derm]:
            for p in m.parameters():
                p.requires_grad = False
            for p in m.features[7].parameters():
                p.requires_grad = True

    def unfreeze_resnets(self):
        for m in [self.resnet_clinic, self.resnet_derm]:
            for p in m.features[6].parameters():
                p.requires_grad = True

    def forward(self, clinic_img, derm_img):
        feat_c = self.resnet_clinic(clinic_img)
        feat_d = self.resnet_derm(derm_img)
        fused, attn_weights = self.fusion(feat_c, feat_d)
        return self.classifier(fused), attn_weights

In [ ]:
class DualBranchSECrossCombined(nn.Module):
    """Full Model: dual SE-ResNet50 + bidirectional cross-attention fusion + classifier."""
    def __init__(self, num_classes=5, embed_dim=256, num_heads=4, dropout=0.32,
                 pretrained=True):
        super().__init__()
        self.resnet_clinic = SEResNet50Backbone(pretrained=pretrained)
        self.resnet_derm   = SEResNet50Backbone(pretrained=pretrained)
        self.fusion        = BidirectionalCrossAttnFusion(2048, embed_dim, num_heads, dropout)
        self.classifier = nn.Sequential(
            nn.Linear(embed_dim, embed_dim // 2),
            nn.BatchNorm1d(embed_dim // 2),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(embed_dim // 2, num_classes),
        )

    def _freeze_backbones(self):
        for m in [self.resnet_clinic, self.resnet_derm]:
            for p in m.parameters():
                p.requires_grad = False
            for p in m.features[7].parameters():
                p.requires_grad = True

    def unfreeze_resnets(self):
        for m in [self.resnet_clinic, self.resnet_derm]:
            for p in m.features[6].parameters():
                p.requires_grad = True

    def forward(self, clinic_img, derm_img):
        feat_c = self.resnet_clinic(clinic_img)
        feat_d = self.resnet_derm(derm_img)
        fused, attn_weights = self.fusion(feat_c, feat_d)
        return self.classifier(fused), attn_weights

## 3 — Improved shared trainer

In [ ]:
def validate(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for clinic_imgs, derm_imgs, labels_batch in loader:
            clinic_imgs  = clinic_imgs.to(device)
            derm_imgs    = derm_imgs.to(device)
            labels_batch = labels_batch.to(device)
            logits, _    = model(clinic_imgs, derm_imgs)
            loss         = criterion(logits, labels_batch)
            total_loss  += loss.item() * clinic_imgs.size(0)
            correct     += logits.argmax(1).eq(labels_batch).sum().item()
            total      += clinic_imgs.size(0)
    return total_loss / total, correct / total


def warmup_cosine_lambda(warmup_epochs, phase_epochs):
    def lr_lambda(ep):
        if warmup_epochs and ep < warmup_epochs:
            return (ep + 1) / warmup_epochs
        prog = (ep - warmup_epochs) / max(1, phase_epochs - warmup_epochs)
        return 0.5 * (1 + math.cos(math.pi * prog))
    return lr_lambda


@torch.no_grad()
def val_f1_macro(model, loader):
    model.eval()
    preds, ys = [], []
    for clinic, derm, y in loader:
        out = model(clinic.to(device), derm.to(device))
        logits = out[0] if isinstance(out, tuple) else out
        preds.extend(logits.argmax(1).cpu().numpy()); ys.extend(y.numpy())
    return f1_score(np.array(ys), np.array(preds), average='macro', zero_division=0)


def nt_xent_loss(z_clinic, z_derm, temperature=0.5):
    B = z_clinic.size(0)
    z = torch.cat([z_clinic, z_derm], dim=0)
    sim = torch.mm(z, z.t()) / temperature
    sim.masked_fill_(torch.eye(2 * B, dtype=torch.bool, device=z.device), float('-inf'))
    targets = (torch.arange(2 * B, device=z.device) + B) % (2 * B)
    return F.cross_entropy(sim, targets)


def update_bn_dual(loader, model):
    """Recompute BatchNorm running stats for a two-input model (torch update_bn is single-input)."""
    momenta = {}
    for mod in model.modules():
        if isinstance(mod, nn.modules.batchnorm._BatchNorm):
            mod.reset_running_stats(); momenta[mod] = mod.momentum; mod.momentum = None
    if not momenta:
        return
    was = model.training; model.train()
    with torch.no_grad():
        for clinic, derm, _ in loader:
            model(clinic.to(device), derm.to(device))
    for bn, m in momenta.items():
        bn.momentum = m
    model.train(was)


def run_training_improved(model, ckpt_path, tag, train_loader, val_loader, loss_weights, hp,
                          total_epochs=FINAL_EPOCHS, patience=PATIENCE, use_swa=True,
                          trial=None, verbose=True):
    """Shared improved trainer for all 7 models. hp=dict(lr, weight_decay, dropout, label_smoothing
    [, contrastive_lambda, contrastive_temp]). AdamW + warmup->cosine + label-smoothed weighted CE
    + SWA + best-by-val-F1 + AMP. If `trial` given, reports val-F1 each epoch for Optuna pruning."""
    model = model.to(device); model._freeze_backbones()
    lr, wd, ls = hp['lr'], hp['weight_decay'], hp['label_smoothing']
    criterion = nn.CrossEntropyLoss(weight=loss_weights, label_smoothing=ls)
    scaler = torch.amp.GradScaler(enabled=USE_AMP)

    def build_os(params, base_lr, phase_ep, warmup):
        opt = optim.AdamW(list(params), lr=base_lr, weight_decay=wd)
        if warmup and warmup > 0:
            sched = optim.lr_scheduler.LambdaLR(opt, warmup_cosine_lambda(warmup, phase_ep))
        else:
            sched = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(1, phase_ep))
        return opt, sched
    opt, sched = build_os(filter(lambda p: p.requires_grad, model.parameters()),
                          lr, UNFREEZE_EPOCH, WARMUP_EPOCHS)

    swa_model = AveragedModel(model) if use_swa else None
    swa_start = int(SWA_START_FRAC * total_epochs); swa_sched = None; swa_n = 0
    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': [], 'val_f1': []}
    best_f1, best_state, early_counter, early_stop_epoch = -1.0, None, 0, None
    start = time.time()

    for epoch in range(total_epochs):
        if epoch == UNFREEZE_EPOCH:
            model.unfreeze_resnets()
            opt, sched = build_os(model.parameters(), lr / 4, total_epochs - UNFREEZE_EPOCH, 0)
            if use_swa:
                swa_sched = SWALR(opt, swa_lr=lr / 8)

        model.train(); tl = tc = tn = 0.0
        for clinic, derm, y in train_loader:
            clinic, derm, y = clinic.to(device), derm.to(device), y.to(device)
            opt.zero_grad()
            with torch.autocast(device_type='cuda', enabled=USE_AMP):
                out = model(clinic, derm)
                logits = out[0] if isinstance(out, tuple) else out
                loss = criterion(logits, y)
                clam = hp.get('contrastive_lambda', 0.0)
                if clam > 0 and hasattr(model, 'project'):
                    zc, zd = model.project(clinic, derm)
                    loss = loss + clam * nt_xent_loss(zc, zd, hp.get('contrastive_temp', 0.5))
            if USE_AMP:
                scaler.scale(loss).backward(); scaler.unscale_(opt)
                nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
                scaler.step(opt); scaler.update()
            else:
                loss.backward(); nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP); opt.step()
            tl += loss.item() * clinic.size(0); tc += logits.argmax(1).eq(y).sum().item(); tn += clinic.size(0)
        tr_loss, tr_acc = tl / tn, tc / tn
        va_loss, va_acc = validate(model, val_loader, criterion)
        va_f1 = val_f1_macro(model, val_loader)

        in_swa = use_swa and epoch >= swa_start
        if in_swa:
            swa_model.update_parameters(model); swa_n += 1
            if swa_sched is not None: swa_sched.step()
        else:
            sched.step()

        history['train_loss'].append(tr_loss); history['val_loss'].append(va_loss)
        history['train_acc'].append(tr_acc); history['val_acc'].append(va_acc)
        history['val_f1'].append(va_f1)

        if va_f1 > best_f1:
            best_f1, best_state, early_counter = va_f1, copy.deepcopy(model.state_dict()), 0
        else:
            early_counter += 1
        if verbose:
            tag2 = 'SWA' if in_swa else f'es {early_counter}/{patience}'
            print(f'  Ep {epoch+1:2d}/{total_epochs} | train {tr_loss:.4f}/{tr_acc:.4f} | '
                  f'val {va_loss:.4f}/{va_acc:.4f} | f1 {va_f1:.4f} | {tag2}')
        if trial is not None:
            trial.report(va_f1, epoch)
            if trial.should_prune():
                raise optuna.TrialPruned()
        if (not in_swa) and early_counter >= patience:
            early_stop_epoch = epoch + 1
            if verbose: print(f'  -> early stop {early_stop_epoch}')
            break

    candidates = []
    if best_state is not None:
        model.load_state_dict(best_state); candidates.append(('best_f1', model, best_f1))
    if use_swa and swa_n > 0:
        update_bn_dual(train_loader, swa_model)
        candidates.append(('swa', swa_model.module, val_f1_macro(swa_model.module, val_loader)))
    name, chosen, chosen_f1 = max(candidates, key=lambda c: c[2])
    save_state = copy.deepcopy(chosen.state_dict())
    torch.save({'model_state_dict': save_state, 'class_names': class_names, 'label_map': label_map,
                'history': history, 'best_val_f1': chosen_f1, 'selected': name,
                'early_stop_epoch': early_stop_epoch, 'hp': hp}, ckpt_path)
    if verbose:
        print(f'{tag}: {(time.time()-start)/60:.1f} min | selected {name} | val f1 {chosen_f1:.4f} | saved {ckpt_path}')
    return history


## 4 — Shared hyperparameter search (Full Model)

ONE Optuna search tunes lr / weight_decay / dropout / label_smoothing on the Full Model
(the thesis contribution). Best HPs are locked to JSON and reused by ALL 7 models so the
architecture comparison stays fair. Resumable SQLite study; warm-started with the
already-tuned baseline. Set SEARCH=False to skip and reuse a previously locked JSON.

In [ ]:
SMOKE = globals().get('SMOKE', False)
SEARCH        = True          # False -> load existing bestparams JSON, skip search
N_TRIALS      = 2 if SMOKE else 15
SEARCH_EPOCHS = 2 if SMOKE else 25
SEARCH_PAT    = 10
STUDY_DB      = f'sqlite:///optuna_singular_{DATASET_NAME}.db'
STUDY_NAME    = f'singular_improved_{DATASET_NAME}'
BESTPARAMS    = f'singular_bestparams_{DATASET_NAME}.json'

WARM_START = {'lr': 0.00075, 'weight_decay': 0.000957, 'dropout': 0.32, 'label_smoothing': 0.1}

def hp_objective(trial):
    hp = {
        'lr':              trial.suggest_float('lr', 1e-4, 3e-3, log=True),
        'weight_decay':    trial.suggest_float('weight_decay', 1e-5, 3e-3, log=True),
        'dropout':         trial.suggest_float('dropout', 0.2, 0.5),
        'label_smoothing': trial.suggest_float('label_smoothing', 0.0, 0.15),
    }
    tr, va, _, _ = make_loaders(BATCH_SIZE, ACTIVE_DS)
    model = DualBranchSECrossCombined(num_classes=NC, dropout=hp['dropout'], pretrained=True)
    hist = run_training_improved(model, f'/tmp/_search_{DATASET_NAME}.pth', f'trial{trial.number}',
                                 tr, va, ACTIVE_DS['loss_weights'], hp,
                                 total_epochs=SEARCH_EPOCHS, patience=SEARCH_PAT,
                                 use_swa=True, trial=trial, verbose=False)
    del model
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return max(hist['val_f1'])

if SEARCH:
    study = optuna.create_study(study_name=STUDY_NAME, storage=STUDY_DB, direction='maximize',
                                sampler=TPESampler(seed=SEED, multivariate=True),
                                pruner=MedianPruner(n_startup_trials=5, n_warmup_steps=8),
                                load_if_exists=True)
    if len(study.trials) == 0:
        study.enqueue_trial(WARM_START)
    remaining = max(0, N_TRIALS - len(study.trials))
    print(f'{STUDY_NAME}: {len(study.trials)} trials done, running {remaining} more...')
    if remaining > 0:
        study.optimize(hp_objective, n_trials=remaining, gc_after_trial=True)
    BEST_HP = study.best_params
    json.dump({'dataset': DATASET_NAME, 'best_val_f1': study.best_value, 'best_hp': BEST_HP},
              open(BESTPARAMS, 'w'), indent=2)
    print(f'Best val f1 {study.best_value:.4f} | {BEST_HP} | saved {BESTPARAMS}')
else:
    BEST_HP = json.load(open(BESTPARAMS))['best_hp']
    print(f'Loaded locked HPs: {BEST_HP}')

## 5 — Train all 7 with locked HPs

In [ ]:
# Single-RGB
ckpt = f'thesis_singlebranch_rgb_improved_{DATASET_NAME}_best.pth'
if Path(ckpt).exists():
    print(f'[skip] Single-RGB: {ckpt} exists')
else:
    tr_loader, va_loader, _, _ = make_loaders(BATCH_SIZE, ACTIVE_DS)
    _model = SingleBranchRGBClassifier(num_classes=NC, dropout=BEST_HP["dropout"])
    print('\n=== Single-RGB ===')
    run_training_improved(_model, ckpt, 'Single-RGB', tr_loader, va_loader,
                          ACTIVE_DS['loss_weights'], BEST_HP,
                          total_epochs=FINAL_EPOCHS, patience=PATIENCE, use_swa=True)
    del _model
    if torch.cuda.is_available(): torch.cuda.empty_cache()

In [ ]:
# Dual-Branch Concat
ckpt = f'thesis_baseline_concat_improved_{DATASET_NAME}_best.pth'
if Path(ckpt).exists():
    print(f'[skip] Dual-Branch Concat: {ckpt} exists')
else:
    tr_loader, va_loader, _, _ = make_loaders(BATCH_SIZE, ACTIVE_DS)
    _model = DualBranchBaseline(num_classes=NC, dropout=BEST_HP["dropout"])
    print('\n=== Dual-Branch Concat ===')
    run_training_improved(_model, ckpt, 'Dual-Branch Concat', tr_loader, va_loader,
                          ACTIVE_DS['loss_weights'], BEST_HP,
                          total_epochs=FINAL_EPOCHS, patience=PATIENCE, use_swa=True)
    del _model
    if torch.cuda.is_available(): torch.cuda.empty_cache()

In [ ]:
# SE-ResNet Concat
ckpt = f'thesis_se_concat_improved_{DATASET_NAME}_best.pth'
if Path(ckpt).exists():
    print(f'[skip] SE-ResNet Concat: {ckpt} exists')
else:
    tr_loader, va_loader, _, _ = make_loaders(BATCH_SIZE, ACTIVE_DS)
    _model = DualBranchSEResNet(num_classes=NC, dropout=BEST_HP["dropout"])
    print('\n=== SE-ResNet Concat ===')
    run_training_improved(_model, ckpt, 'SE-ResNet Concat', tr_loader, va_loader,
                          ACTIVE_DS['loss_weights'], BEST_HP,
                          total_epochs=FINAL_EPOCHS, patience=PATIENCE, use_swa=True)
    del _model
    if torch.cuda.is_available(): torch.cuda.empty_cache()

In [ ]:
# Add-Fusion
ckpt = f'thesis_addfusion_improved_{DATASET_NAME}_best.pth'
if Path(ckpt).exists():
    print(f'[skip] Add-Fusion: {ckpt} exists')
else:
    tr_loader, va_loader, _, _ = make_loaders(BATCH_SIZE, ACTIVE_DS)
    _model = DualBranchElementwiseFusion(num_classes=NC, dropout=BEST_HP["dropout"], fusion="add")
    print('\n=== Add-Fusion ===')
    run_training_improved(_model, ckpt, 'Add-Fusion', tr_loader, va_loader,
                          ACTIVE_DS['loss_weights'], BEST_HP,
                          total_epochs=FINAL_EPOCHS, patience=PATIENCE, use_swa=True)
    del _model
    if torch.cuda.is_available(): torch.cuda.empty_cache()

In [ ]:
# Mul-Fusion
ckpt = f'thesis_mulfusion_improved_{DATASET_NAME}_best.pth'
if Path(ckpt).exists():
    print(f'[skip] Mul-Fusion: {ckpt} exists')
else:
    tr_loader, va_loader, _, _ = make_loaders(BATCH_SIZE, ACTIVE_DS)
    _model = DualBranchElementwiseFusion(num_classes=NC, dropout=BEST_HP["dropout"], fusion="mul")
    print('\n=== Mul-Fusion ===')
    run_training_improved(_model, ckpt, 'Mul-Fusion', tr_loader, va_loader,
                          ACTIVE_DS['loss_weights'], BEST_HP,
                          total_epochs=FINAL_EPOCHS, patience=PATIENCE, use_swa=True)
    del _model
    if torch.cuda.is_available(): torch.cuda.empty_cache()

In [ ]:
# BiCrossAttn
ckpt = f'thesis_crossattn_improved_{DATASET_NAME}_best.pth'
if Path(ckpt).exists():
    print(f'[skip] BiCrossAttn: {ckpt} exists')
else:
    tr_loader, va_loader, _, _ = make_loaders(BATCH_SIZE, ACTIVE_DS)
    _model = DualBranchBiCrossAttn(num_classes=NC, dropout=BEST_HP["dropout"])
    print('\n=== BiCrossAttn ===')
    run_training_improved(_model, ckpt, 'BiCrossAttn', tr_loader, va_loader,
                          ACTIVE_DS['loss_weights'], BEST_HP,
                          total_epochs=FINAL_EPOCHS, patience=PATIENCE, use_swa=True)
    del _model
    if torch.cuda.is_available(): torch.cuda.empty_cache()

In [ ]:
# Full (SE+Cross)
ckpt = f'thesis_full_se_crossattn_improved_{DATASET_NAME}_best.pth'
if Path(ckpt).exists():
    print(f'[skip] Full (SE+Cross): {ckpt} exists')
else:
    tr_loader, va_loader, _, _ = make_loaders(BATCH_SIZE, ACTIVE_DS)
    _model = DualBranchSECrossCombined(num_classes=NC, dropout=BEST_HP["dropout"])
    print('\n=== Full (SE+Cross) ===')
    run_training_improved(_model, ckpt, 'Full (SE+Cross)', tr_loader, va_loader,
                          ACTIVE_DS['loss_weights'], BEST_HP,
                          total_epochs=FINAL_EPOCHS, patience=PATIENCE, use_swa=True)
    del _model
    if torch.cuda.is_available(): torch.cuda.empty_cache()

## 6 — Grand comparison (7 models, held-out test) + old-vs-improved delta

In [ ]:
# ── Shared comparison helper (used by §2.3, §3.x, §4.x, §5.x, §6) ────────────
def evaluate_checkpoint_quiet(cls, kwargs, ckpt_path, test_loader):
    """Load a checkpoint, evaluate on test_loader, return metrics dict.
    Silent — for use inside aggregate comparisons. Returns None if file missing."""
    if not Path(ckpt_path).exists():
        return None
    model = cls(**kwargs).to(device)
    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
    model.load_state_dict(ckpt['model_state_dict'])
    model.eval()
    preds, y, probs = [], [], []
    with torch.no_grad():
        for clinic_imgs, derm_imgs, lb in test_loader:
            logits, _ = model(clinic_imgs.to(device), derm_imgs.to(device))
            p = F.softmax(logits, dim=1)
            preds.extend(logits.argmax(1).cpu().numpy())
            y.extend(lb.numpy())
            probs.extend(p.cpu().numpy())
    y = np.array(y); preds = np.array(preds); probs = np.array(probs)
    nc = len(class_names)
    metrics = {
        'accuracy':          accuracy_score(y, preds),
        'balanced_accuracy': balanced_accuracy_score(y, preds),
        'f1_macro':          f1_score(y, preds, average='macro',    zero_division=0),
        'f1_weighted':       f1_score(y, preds, average='weighted', zero_division=0),
        'cohen_kappa':       cohen_kappa_score(y, preds),
        'mcc':               matthews_corrcoef(y, preds),
        'log_loss':          log_loss(y, probs, labels=list(range(nc))),
    }
    del model
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return metrics


def render_comparison(results, ds_name, title_prefix, png_stem):
    """Render a metrics table + bar chart for a {variant_name: metrics_dict} dict."""
    if not results:
        print(f'[{ds_name}] No checkpoints found — skipping figure.')
        return None
    df_cmp = pd.DataFrame(results).T
    df_cmp = df_cmp[['accuracy', 'balanced_accuracy', 'f1_macro', 'f1_weighted',
                     'cohen_kappa', 'mcc', 'log_loss']]
    df_cmp.index.name = 'Model'

    fig = plt.figure(figsize=(14, max(7, 2 + 0.6 * len(df_cmp))))
    gs = fig.add_gridspec(2, 1, height_ratios=[len(df_cmp) + 2, 6], hspace=0.30)
    fig.suptitle(f'{title_prefix} — {ds_name} (held-out test set)',
                 fontsize=14, fontweight='bold', y=0.98)

    ax_t = fig.add_subplot(gs[0]); ax_t.axis('off')
    col_labels = ['Accuracy', 'Balanced Acc', 'F1 (macro)', 'F1 (weighted)',
                  'Cohen kappa', 'MCC', 'Log loss']
    cell_text = [[f'{v:.4f}' for v in row] for row in df_cmp.values]
    table = ax_t.table(cellText=cell_text, rowLabels=df_cmp.index, colLabels=col_labels,
                       cellLoc='center', rowLoc='center', loc='center')
    table.auto_set_font_size(False); table.set_fontsize(11); table.scale(1, 1.7)
    for j in range(len(col_labels)):
        table[0, j].set_facecolor('#34495e')
        table[0, j].set_text_props(color='white', fontweight='bold')
    for j, col in enumerate(df_cmp.columns):
        best_i = df_cmp[col].idxmin() if col == 'log_loss' else df_cmp[col].idxmax()
        r = list(df_cmp.index).index(best_i)
        table[r + 1, j].set_facecolor('#d5f5e3')
        table[r + 1, j].set_text_props(fontweight='bold')
    ax_t.set_title('Green = best per metric   ·   Log loss: lower is better',
                   fontsize=9, pad=8)

    ax_b = fig.add_subplot(gs[1])
    metrics_to_plot = ['accuracy', 'balanced_accuracy', 'f1_macro', 'f1_weighted',
                       'cohen_kappa', 'mcc']
    df_cmp[metrics_to_plot].T.plot(kind='bar', ax=ax_b, rot=0, colormap='tab10', width=0.8)
    ax_b.set_ylabel('Score'); ax_b.set_ylim(0, 1)
    ax_b.set_title('Metric comparison (higher is better)', fontsize=11)
    ax_b.set_xticklabels(['Accuracy', 'Balanced Acc', 'F1 (macro)', 'F1 (weighted)',
                          'Cohen kappa', 'MCC'])
    ax_b.legend(title='Model', bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=9)
    ax_b.grid(alpha=0.3, axis='y')

    out_png = f'{png_stem}_{ds_name}.png'
    plt.savefig(out_png, dpi=200, bbox_inches='tight')
    print(f'[{ds_name}] Saved -> {out_png}')
    plt.show()
    return df_cmp


def mini_compare(variants, title, png_stem):
    """Evaluate each variant on ACTIVE_DS; render one figure; return {label: metrics}."""
    ds = ACTIVE_DS
    print(f'\n--- {title} · {ds["name"]} ---')
    _, _, test_loader, _ = make_loaders(BATCH_SIZE, ds)
    ds_results = {}
    for label, cls, kwargs, stem in variants:
        ckpt_path = f'{stem}_{ds["name"]}_best.pth'
        m = evaluate_checkpoint_quiet(cls, kwargs, ckpt_path, test_loader)
        if m is None:
            print(f'  {label}: {ckpt_path} not found — skipping')
            continue
        ds_results[label] = m
        print(f'  {label:<28s}  acc={m["accuracy"]:.4f}  '
              f'f1_macro={m["f1_macro"]:.4f}  kappa={m["cohen_kappa"]:.4f}')
    render_comparison(ds_results, ds['name'], title, png_stem)
    return ds_results


print('Comparison helpers ready: mini_compare(variants, title, png_stem) — runs on ACTIVE_DS.')


In [ ]:
# Evaluate all 7 improved checkpoints on the held-out test set.
EVAL_SPECS = [
    ('Single-RGB',         SingleBranchRGBClassifier,   dict(num_classes=NC, dropout=BEST_HP['dropout']),                 'thesis_singlebranch_rgb'),
    ('Dual-Branch Concat', DualBranchBaseline,          dict(num_classes=NC, dropout=BEST_HP['dropout']),                 'thesis_baseline_concat'),
    ('SE-ResNet Concat',   DualBranchSEResNet,          dict(num_classes=NC, dropout=BEST_HP['dropout']),                 'thesis_se_concat'),
    ('Add-Fusion',         DualBranchElementwiseFusion, dict(num_classes=NC, dropout=BEST_HP['dropout'], fusion='add'),   'thesis_addfusion'),
    ('Mul-Fusion',         DualBranchElementwiseFusion, dict(num_classes=NC, dropout=BEST_HP['dropout'], fusion='mul'),   'thesis_mulfusion'),
    ('BiCrossAttn',        DualBranchBiCrossAttn,       dict(num_classes=NC, dropout=BEST_HP['dropout']),                 'thesis_crossattn'),
    ('Full (SE+Cross)',    DualBranchSECrossCombined,   dict(num_classes=NC, dropout=BEST_HP['dropout']),                 'thesis_full_se_crossattn'),
]
_, _, test_loader, _ = make_loaders(BATCH_SIZE, ACTIVE_DS)
results = {}
for label, cls, kwargs, stem in EVAL_SPECS:
    kwargs = dict(kwargs); kwargs['pretrained'] = False
    ckpt = f'{stem}_improved_{DATASET_NAME}_best.pth'
    m = evaluate_checkpoint_quiet(cls, kwargs, ckpt, test_loader)
    if m is None:
        print(f'  {label}: {ckpt} missing -- skipped'); continue
    results[label] = m
    print(f'  {label:<20s} acc={m["accuracy"]:.4f} f1_macro={m["f1_macro"]:.4f} kappa={m["cohen_kappa"]:.4f}')

df_cmp = render_comparison(results, DATASET_NAME, 'Improved 7-model comparison', 'comparison_improved')
if df_cmp is not None:
    df_cmp.to_csv(f'comparison_improved_{DATASET_NAME}.csv')
    print(f'Saved comparison_improved_{DATASET_NAME}.csv')

In [ ]:
# Old-vs-improved delta (reads the baseline notebook's comparison CSV if present).
old_csv = f'comparison_metrics_{DATASET_NAME}.csv'
if Path(old_csv).exists() and df_cmp is not None:
    old = pd.read_csv(old_csv, index_col=0)
    common = [i for i in df_cmp.index if i in old.index]
    if common:
        delta = (df_cmp.loc[common, ['accuracy', 'f1_macro', 'cohen_kappa']]
                 - old.loc[common, ['accuracy', 'f1_macro', 'cohen_kappa']])
        delta.columns = ['d_acc', 'd_f1_macro', 'd_kappa']
        print('Improved minus baseline:')
        print(delta.round(4).to_string())
        delta.to_csv(f'delta_improved_{DATASET_NAME}.csv')
    else:
        print('(no overlapping model names with baseline CSV)')
else:
    print(f'(no baseline {old_csv} found -- skipping delta)')